# Chapter 8 — Regression Methods II: Polynomial Regression, Overfitting & Ridge Regression

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winnertechserv/aimlgybs/blob/main/part2_classic_ml/ch08_regression_methods_ii/ch08_notebook.ipynb)

Companion notebook for **Chapter 8** of *AI-Machine Learning: Get Your Base Strong*. Read the chapter first, then run each cell and compare your output with the output printed in the book (shown below each cell).


In [ ]:
# Optional: install the exact library versions used for the outputs in the book.
# Google Colab already has these libraries; newer versions may change a few digits.
# !pip install -q numpy==2.5.3 pandas==3.0.5 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2 pillow==12.3.0

## Key takeaways

- Polynomial regression fits curves but is still linear in the weights, so least squares solves it in closed form.
- The design matrix Φ holds the powers (monomials) of the inputs, and w = (ΦᵀΦ)⁻¹Φᵀy.
- Choose the degree M with a validation or test set, never with the training error.
- Overfitting means low training error, high test error and huge weights.
- More data and L2 regularization (ridge regression) both reduce overfitting.
- With many inputs the number of polynomial terms explodes: D = (d+M)!/(d!M!).

## Example 1: Polynomial Curve Fitting (Degrees 1, 2 and 3)

Humidity is predicted from temperature with polynomials of degree 1, 2 and 3, and each model is tested on five days it never saw. np.vander builds the design matrix (columns 1, x, x², ...) and the pseudo-inverse solves for the weights. The straight line misses the curve (test RMSE 3.56). Degree 2 follows it closely (0.60). Degree 3 fits the training days slightly better (0.76 against 0.84) but does worse on the new days (0.86). That is an early sign of overfitting, which Example 2 explores fully.

In [1]:
# Chapter 8 - Example 1: Polynomial curve fitting - humidity from temperature
# Degree M = 1, 2, 3 using the design (Vandermonde) matrix and the pseudo-inverse.
import numpy as np
np.set_printoptions(suppress=True)

temp = np.array([29.3, 29.3, 25.2, 22.0, 18.8, 23.4, 23.7, 18.6, 18.7, 32.0,
                 27.1, 21.3, 24.1, 31.6, 30.6])
hum = np.array([45.3, 46.0, 53.7, 62.7, 81.8, 59.8, 57.2, 82.4, 81.5, 43.5,
                48.4, 68.5, 55.5, 42.9, 42.8])
# Five new days the model never saw during training
t_test = np.array([19.5, 22.8, 26.0, 28.4, 31.2])
h_test = np.array([77.5, 61.4, 50.5, 44.8, 42.8])

for M in (1, 2, 3):
    Phi = np.vander(temp, M + 1, increasing=True)      # columns: 1, x, x^2, ..., x^M
    w = np.linalg.pinv(Phi) @ hum                      # w = (Phi^T Phi)^-1 Phi^T y
    train_rmse = np.sqrt(np.mean((Phi @ w - hum) ** 2))
    pred = np.vander(t_test, M + 1, increasing=True) @ w
    test_rmse = np.sqrt(np.mean((pred - h_test) ** 2))
    print(f"M = {M}: Phi is {Phi.shape[0]}x{Phi.shape[1]}, w = {np.round(w, 4)}")
    print(f"       training RMSE = {train_rmse:.2f}   test RMSE (5 new days) = {test_rmse:.2f}")
    print(f"       predictions {np.round(pred, 1)}  vs actual {h_test}\n")
print("M = 2 fits the curve; M = 3 adds flexibility the data does not need.")

M = 1: Phi is 15x2, w = [131.7842  -2.9405]
       training RMSE = 4.02   test RMSE (5 new days) = 3.56
       predictions [74.4 64.7 55.3 48.3 40. ]  vs actual [77.5 61.4 50.5 44.8 42.8]

M = 2: Phi is 15x3, w = [271.97   -14.444    0.2282]
       training RMSE = 0.84   test RMSE (5 new days) = 0.60
       predictions [77.1 61.3 50.7 45.9 43.5]  vs actual [77.5 61.4 50.5 44.8 42.8]

M = 3: Phi is 15x4, w = [364.8974 -26.0419   0.7014  -0.0063]
       training RMSE = 0.76   test RMSE (5 new days) = 0.86
       predictions [77.  60.9 51.  46.4 43.4]  vs actual [77.5 61.4 50.5 44.8 42.8]

M = 2 fits the curve; M = 3 adds flexibility the data does not need.


## Example 2: Watching Overfitting Happen

We take 10 noisy points from a sine wave and fit polynomials from degree 0 to 9. Read the table from top to bottom. Training error keeps falling, all the way to 0 at M = 9, but test error is lowest around M = 3 and then climbs again. The largest weight explodes to about 95,000, which is the warning sign. The second table fixes the degree at 9 and adds data: with 15 or 100 points the test error drops back to about 0.32.

In [2]:
# Chapter 8 - Example 2: Model complexity vs overfitting
# Fit polynomials of degree M to 10 noisy points from a sine wave, then test on new points.
import numpy as np

rng = np.random.default_rng(0)
f = lambda x: np.sin(2 * np.pi * x)            # the true (hidden) curve

def make(n, even=True):
    x = np.linspace(0, 1, n) if even else rng.uniform(0, 1, n)   # training x evenly spaced
    return x, f(x) + rng.normal(0, 0.3, n)

def fit_rmse(x_tr, y_tr, x_te, y_te, M):
    Phi = np.vander(x_tr, M + 1, increasing=True)
    w = np.linalg.lstsq(Phi, y_tr, rcond=None)[0]
    r = lambda x, y: np.sqrt(np.mean((np.vander(x, M + 1, increasing=True) @ w - y) ** 2))
    return r(x_tr, y_tr), r(x_te, y_te), np.abs(w).max()

x_te, y_te = make(500, even=False)
x10, y10 = make(10)
print("N = 10 training points")
print(" M   train RMSE   test RMSE   largest |w|")
for M in (0, 1, 3, 6, 9):
    tr, te, wmax = fit_rmse(x10, y10, x_te, y_te, M)
    print(f" {M}     {tr:.3f}       {te:8.3f}   {wmax:12.1f}")
print("M = 9 passes through all 10 points (train error ~0) but is much worse on new data.")

print("\nMore data tames the same complex model (M = 9):")
for n in (10, 15, 100):
    x, y = (x10, y10) if n == 10 else make(n)
    tr, te, _ = fit_rmse(x, y, x_te, y_te, 9)
    print(f"  N = {n:3d}: train RMSE {tr:.3f}, test RMSE {te:8.3f}")

N = 10 training points
 M   train RMSE   test RMSE   largest |w|
 0     0.672          0.751            0.0
 1     0.575          0.572            1.1
 3     0.265          0.324           28.6
 6     0.249          0.332          917.3
 9     0.000          0.772        95188.5
M = 9 passes through all 10 points (train error ~0) but is much worse on new data.

More data tames the same complex model (M = 9):
  N =  10: train RMSE 0.000, test RMSE    0.772
  N =  15: train RMSE 0.108, test RMSE    0.321
  N = 100: train RMSE 0.293, test RMSE    0.324


## Example 3: Ridge Regression and Counting Monomials

We keep the overfitting degree-9 model and add an L2 penalty λ. With λ = 0 the test RMSE is 0.97 (not the 0.77 of Example 2, because this example draws a fresh set of noisy points with a different random seed) and the weights are in the millions. A tiny λ of 0.0001 cuts the test error to 0.35 and the weight size to about 20. A large λ of 1 overshoots and underfits. The second part checks the formula D = (d+M)!/(d!M!), including the 364 terms for 11 inputs at degree 3, and builds φ([2, 10]).

In [3]:
# Chapter 8 - Example 3: Ridge regression (L2 regularization) and polynomial surfaces
import numpy as np
from math import factorial

rng = np.random.default_rng(2)
f = lambda x: np.sin(2 * np.pi * x)
x_tr = np.linspace(0, 1, 10); y_tr = f(x_tr) + rng.normal(0, 0.3, 10)
x_te = rng.uniform(0, 1, 500); y_te = f(x_te) + rng.normal(0, 0.3, 500)

M = 9
Phi = np.vander(x_tr, M + 1, increasing=True)
Phi_te = np.vander(x_te, M + 1, increasing=True)
print("Degree M = 9, N = 10 points. Ridge: w = (Phi^T Phi + lambda I)^-1 Phi^T y")
print(" lambda      train RMSE  test RMSE  size of w (||w||)")
for lam in (0, 1e-6, 1e-4, 1e-2, 1):
    if lam == 0:
        w = np.linalg.lstsq(Phi, y_tr, rcond=None)[0]       # plain least squares
    else:
        w = np.linalg.solve(Phi.T @ Phi + lam * np.eye(M + 1), Phi.T @ y_tr)
    tr = np.sqrt(np.mean((Phi @ w - y_tr) ** 2))
    te = np.sqrt(np.mean((Phi_te @ w - y_te) ** 2))
    print(f" {lam:<9g}   {tr:.3f}      {te:8.3f}   {np.linalg.norm(w):12.1f}")
print("A small lambda shrinks the weights and fixes overfitting; too large underfits.")

# How many terms does a polynomial in d inputs of degree M have?  D = (d+M)! / (d! M!)
print("\nNumber of monomials D = (d+M)!/(d! M!):")
for d, Mdeg in ((2, 2), (2, 3), (3, 2), (11, 3)):
    D = factorial(d + Mdeg) // (factorial(d) * factorial(Mdeg))
    print(f"  d = {d:2d} inputs, degree M = {Mdeg}: D = {D}")

# Monomials for x = [2, 10], M = 2: 1, x1, x2, x1^2, x2^2, x1*x2
x1, x2 = 2, 10
print("\nphi([2, 10]) for M = 2:", [1, x1, x2, x1**2, x2**2, x1 * x2])

Degree M = 9, N = 10 points. Ridge: w = (Phi^T Phi + lambda I)^-1 Phi^T y
 lambda      train RMSE  test RMSE  size of w (||w||)
 0           0.000         0.965      1219209.9
 1e-06       0.254         0.362          117.4
 0.0001      0.267         0.350           19.5
 0.01        0.326         0.440            4.1
 1           0.471         0.620            0.6
A small lambda shrinks the weights and fixes overfitting; too large underfits.

Number of monomials D = (d+M)!/(d! M!):
  d =  2 inputs, degree M = 2: D = 6
  d =  2 inputs, degree M = 3: D = 10
  d =  3 inputs, degree M = 2: D = 10
  d = 11 inputs, degree M = 3: D = 364

phi([2, 10]) for M = 2: [1, 2, 10, 4, 100, 20]
